# Notebook Expérimental : Détection de Churn Client (À Refactoriser)
Ce notebook a été codé rapidement par un stagiaire Data Scientist.
Il comporte de multiples anti-patterns (chemins en dur, variables globales, pas de fonctions, pas de tests).
**Votre mission de TP :** Le transformer en package industriel `src/` propre, testé et modulaire !

In [1]:
# Cellule 1 : Imports en vrac
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import pickle
import os

In [2]:
# Cellule 2 : Chargement des données avec chemin en dur (ANTI-PATTERN !)
# ATTENTION : Si le chemin n'existe pas sur votre machine, cela plante immédiatement !
data_path = 'data/raw/customer_churn.csv'
df = pd.read_csv(data_path)
print('Dimensions du dataset :', df.shape)
df.head()

In [3]:
# Cellule 3 : Preprocessing mélangé en vrac sans aucune fonction
# Suppression de l'ID client
df = df.drop(columns=['customerID'])

# Encodage binaire de la cible
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Traitement des variables binaires
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
df['Partner'] = df['Partner'].map({'Yes': 1, 'No': 0})
df['Dependents'] = df['Dependents'].map({'Yes': 1, 'No': 0})
df['PhoneService'] = df['PhoneService'].map({'Yes': 1, 'No': 0})
df['PaperlessBilling'] = df['PaperlessBilling'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding des variables multi-classes
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'], drop_first=True)

# Gestion des valeurs nulles éventuelles
df = df.fillna(df.median())

# Normalisation manuelle des charges
df['MonthlyCharges'] = (df['MonthlyCharges'] - df['MonthlyCharges'].min()) / (df['MonthlyCharges'].max() - df['MonthlyCharges'].min())
df['TotalCharges'] = (df['TotalCharges'] - df['TotalCharges'].min()) / (df['TotalCharges'].max() - df['TotalCharges'].min())

print('Colonnes préparées :', df.columns.tolist())

In [4]:
# Cellule 4 : Séparation Train / Test avec variable globale
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train set:', X_train.shape, 'Test set:', X_test.shape)

In [5]:
# Cellule 5 : Entraînement sans suivi de paramètres ni validation rigoureuse
model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_train, y_train)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds)

print(f'Accuracy: {acc:.4f}')
print(f'F1-Score: {f1:.4f}')

In [6]:
# Cellule 6 : Sauvegarde artisanale du modèle sans métadonnées de versioning
os.makedirs('models', exist_ok=True)
with open('models/model_final.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Modèle sauvegardé dans models/model_final.pkl !')